In [ ]:
# --- 0. Imports and Setup ---
import scanpy as sc
import anndata as ad
import numpy as np
import pandas as pd
import os
import pickle # For saving/loading models
import matplotlib.pyplot as plt
import seaborn as sns
import random # For setting Python's random seed

# Make sure you have these installed: pip install scikit-learn scikit-misc
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_curve,
    auc,
    accuracy_score
)
from sklearn.preprocessing import LabelBinarizer

# Adjust the path to import your scpred_py_final package
import sys
module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path)

# Importing the main ScPredModel class and utilities
from scpred_py_final import ScPredModel # Adjusted to scpred_py_final as per context
from scpred_py_final import _analysis_utils # Assuming _analysis_utils is part of the package


print("--- Setting up environment for reproducibility and file management ---")
# Set random seeds for reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE) # For Python's built-in random module
# Set Scanpy's random state for operations that support it
sc.settings.set_figure_params(dpi=80, facecolor='white', figsize=(8, 8)) # Set initial figure params

# Define paths for saving
MODELS_DIR = '../models'
PREPROCESSED_DATA_DIR = '../data/preprocessed'

# Create directories if they don't exist
print(f"Ensuring directories exist: {MODELS_DIR} and {PREPROCESSED_DATA_DIR}")
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(PREPROCESSED_DATA_DIR, exist_ok=True)
print("Directories ensured.")


# --- 1. Data Loading and Initial Annotation ---
print("\n--- Step 1: Loading and Annotating Data ---")
adata = sc.datasets.pbmc3k()
adata.var_names_make_unique()

print(f"Initial raw AnnData object shape: {adata.shape}")
print(f"Initial raw AnnData .obs keys: {adata.obs.keys().tolist()}")
print(f"Initial raw AnnData .var keys: {adata.var.keys().tolist()}\n")


# Generate 'cell_type' labels using a temporary preprocessing pipeline
# This prepares the `adata` object with raw counts and the `cell_type` column
print("Generating temporary cell type labels via Louvain clustering...")
temp_adata_for_labels = adata.copy()
sc.pp.filter_cells(temp_adata_for_labels, min_genes=200)
sc.pp.filter_genes(temp_adata_for_labels, min_cells=3)
sc.pp.normalize_total(temp_adata_for_labels, target_sum=1e4)
sc.pp.log1p(temp_adata_for_labels)
sc.pp.highly_variable_genes(temp_adata_for_labels, min_mean=0.0125, max_mean=3, min_disp=0.5, flavor='seurat')
temp_adata_hvg_for_labels = temp_adata_for_labels[:, temp_adata_for_labels.var.highly_variable].copy()
sc.pp.scale(temp_adata_hvg_for_labels, max_value=10)
sc.tl.pca(temp_adata_hvg_for_labels, svd_solver='arpack', random_state=RANDOM_STATE) # Added random_state
sc.pp.neighbors(temp_adata_hvg_for_labels, n_neighbors=10, n_pcs=40, random_state=RANDOM_STATE) # Added random_state
sc.tl.louvain(temp_adata_hvg_for_labels, random_state=RANDOM_STATE, key_added='cell_type') # Added random_state

# Assign the generated cell_type labels back to the original raw adata object
adata.obs['cell_type'] = temp_adata_hvg_for_labels.obs['cell_type'].reindex(adata.obs_names).astype('category') # Ensure category type

print(f"Full dataset (raw counts + Louvain labels) shape: {adata.shape}")
print(f"Full dataset .obs keys after labeling: {adata.obs.keys().tolist()}")
print("Cell type distribution (all data):\n", adata.obs['cell_type'].value_counts())


# --- 2. Train-Test Split (Stratified) ---
# We split the raw data (with labels) into reference and query sets.
print("\n--- Step 2: Splitting Data (Stratified) ---")
indices = range(adata.n_obs)

ref_idx, query_idx = train_test_split(
    indices,
    test_size=0.3,
    random_state=RANDOM_STATE, # Ensuring split is reproducible
    stratify=adata.obs['cell_type'] # CRITICAL for balanced classes
)

# These AnnData objects contain raw counts and cell_type labels.
# The ScPredModel will handle their internal preprocessing.
ref_adata_raw = adata[ref_idx, :].copy()
query_adata_raw = adata[query_idx, :].copy()

print(f"Reference data (raw + labels) shape: {ref_adata_raw.shape}")
print(f"Query data (raw + labels) shape: {query_adata_raw.shape}")
print("\nReference cell type distribution:\n", ref_adata_raw.obs['cell_type'].value_counts())
print("\nQuery cell type distribution:\n", query_adata_raw.obs['cell_type'].value_counts())


# Calculate and report class imbalance
print("\n--- Class Imbalance Analysis ---")
print("Reference data class imbalance (proportion):\n", ref_adata_raw.obs['cell_type'].value_counts(normalize=True).sort_index())
print("Query data class imbalance (proportion):\n", query_adata_raw.obs['cell_type'].value_counts(normalize=True).sort_index())


# --- 3. Training the scPred Model (with Optimal Parameters) ---
# The ScPredModel will now handle all preprocessing (normalization, log1p, HVG, scaling, PCA) internally.
print("\n--- Step 3: Training scPred Model with Optimal Parameters ---")
scpred_model = ScPredModel()

# Applying optimal parameters from comparative analysis: RBF, C=1.0, Balanced, 1000 HVGs
scpred_model.train(
    ref_adata=ref_adata_raw, # Pass the raw reference data
    cell_type_key='cell_type',
    n_components=30,
    hvg_n_top_genes=1000, # Optimal from comparative analysis
    hvg_flavor='seurat',
    svm_kernel='rbf',     # Optimal from comparative analysis
    svm_c=1.0,            # Optimal from comparative analysis
    svm_random_state=RANDOM_STATE, # Ensuring SVM training is reproducible
    svm_class_weight='balanced' # Optimal from comparative analysis
)

print("\nModel Trained with Optimal Parameters!")
print(f"Scaler: {scpred_model.scaler_}")
print(f"PCA Model: {scpred_model.pca_model_}")
print(f"Classifier: {scpred_model.classifier_}")
print(f"Reference HVGs learned: {len(scpred_model.reference_hvg_genes_)} genes")

# Save the trained ScPredModel object
model_path = os.path.join(MODELS_DIR, 'scpred_model_pbmc3k_optimal.pkl')
scpred_model.save(model_path)
print(f"Optimal ScPredModel saved successfully to: {model_path}")

# Save the preprocessed reference data (now accessible via scpred_model.ref_adata_processed_)
if hasattr(scpred_model, 'ref_adata_processed_') and scpred_model.ref_adata_processed_ is not None:
    preprocessed_ref_path = os.path.join(PREPROCESSED_DATA_DIR, 'pbmc3k_ref_preprocessed_optimal.h5ad')
    scpred_model.ref_adata_processed_.write(preprocessed_ref_path)
    print(f"Preprocessed reference data saved to: {preprocessed_ref_path}")
else:
    print("Warning: scpred_model.ref_adata_processed_ is None. Preprocessed reference data not saved.")


# ==============================================================================
# THOROUGH ANALYSIS: SCENARIO 1 - NO THRESHOLD (threshold=0.0)
# ==============================================================================
print("\n\n======== THOROUGH ANALYSIS: SCENARIO 1 - NO THRESHOLD (threshold=0.0) ========")

# --- 4a. Predicting with the scPred Model (Threshold = 0.0) ---
print("\n--- Step 4a: Predicting on Query Data (Threshold = 0.0) ---")
query_adata_pred_no_threshold = scpred_model.predict(query_adata_raw.copy(), threshold=0.0) # Pass a copy

print("\nQuery Data with Predictions (No Threshold, first 5 rows):")
print(query_adata_pred_no_threshold.obs[['cell_type', 'scpred_prediction']].head())
print(f"\nQuery data .obs keys after prediction: {query_adata_pred_no_threshold.obs.keys().tolist()}")
print(f"Query data .obsm keys after prediction: {list(query_adata_pred_no_threshold.obsm.keys())}")
print("\nPredicted label distribution (No Threshold):\n", query_adata_pred_no_threshold.obs['scpred_prediction'].value_counts(dropna=False))

# Save the preprocessed query data with predictions
preprocessed_query_no_threshold_path = os.path.join(PREPROCESSED_DATA_DIR, 'pbmc3k_query_pred_no_threshold.h5ad')
query_adata_pred_no_threshold.write(preprocessed_query_no_threshold_path)
print(f"Preprocessed query data with predictions (No Threshold) saved to: {preprocessed_query_no_threshold_path}")


# --- 5a. Evaluating Predictions (Threshold = 0.0) ---
true_labels_no_threshold = query_adata_pred_no_threshold.obs['cell_type']
predicted_labels_no_threshold = query_adata_pred_no_threshold.obs['scpred_prediction']

# Filter out NaN predictions for evaluation to avoid ValueError
valid_cells_no_threshold = predicted_labels_no_threshold.notna()
true_labels_no_threshold_filtered = true_labels_no_threshold[valid_cells_no_threshold]
predicted_labels_no_threshold_filtered = predicted_labels_no_threshold[valid_cells_no_threshold]

pred_prob_cols_no_threshold = [col for col in query_adata_pred_no_threshold.obs.columns if col.startswith('scpred_prob_')]
y_pred_probs_df_no_threshold = None
if len(pred_prob_cols_no_threshold) > 0:
    y_pred_probs_df_no_threshold = query_adata_pred_no_threshold.obs.loc[valid_cells_no_threshold, pred_prob_cols_no_threshold].copy()
    y_pred_probs_df_no_threshold = y_pred_probs_df_no_threshold.fillna(0.0)


print("\n--- Evaluation for No Threshold Model ---")
metrics_results_no_threshold = _analysis_utils.evaluate_and_report_metrics(
    true_labels=true_labels_no_threshold_filtered,
    predicted_labels=predicted_labels_no_threshold_filtered,
    classifier_classes=scpred_model.classifier_.classes_,
    y_pred_probs=y_pred_probs_df_no_threshold
)


# --- 6a. Visualizing Results (Threshold = 0.0) ---
print("\n--- Step 6a: Visualizing Results (Threshold = 0.0) ---")

# For UMAP plots, we need the AnnData object to be filtered to only valid cells or handle NaNs in plots.
query_adata_pred_no_threshold_filtered_for_umap = query_adata_pred_no_threshold[valid_cells_no_threshold, :].copy()

if 'iroot' in query_adata_pred_no_threshold_filtered_for_umap.uns:
    del query_adata_pred_no_threshold_filtered_for_umap.uns['iroot']
if 'neighbors' in query_adata_pred_no_threshold_filtered_for_umap.uns:
    del query_adata_pred_no_threshold_filtered_for_umap.uns['neighbors']


if 'X_scpred_pca' in query_adata_pred_no_threshold_filtered_for_umap.obsm and 'X_umap' not in query_adata_pred_no_threshold_filtered_for_umap.obsm:
    print("Computing neighbors and UMAP based on X_scpred_pca for visualization...")
    sc.pp.neighbors(query_adata_pred_no_threshold_filtered_for_umap, n_neighbors=10, use_rep='X_scpred_pca', random_state=RANDOM_STATE)
    sc.tl.umap(query_adata_pred_no_threshold_filtered_for_umap, random_state=RANDOM_STATE)
    print(f"Query data .obsm keys after UMAP: {list(query_adata_pred_no_threshold_filtered_for_umap.obsm.keys())}")
elif 'X_umap' not in query_adata_pred_no_threshold_filtered_for_umap.obsm:
    print("X_scpred_pca or X_umap not found in .obsm. UMAP plots might not be generated.")

# Plot 1: Confusion Matrix (Threshold = 0.0)
print("\n--- Confusion Matrix (Threshold = 0.0) ---")
true_labels_str_no_threshold = true_labels_no_threshold_filtered.astype(str)
predicted_labels_str_no_threshold = predicted_labels_no_threshold_filtered.astype(str)

cm_all_labels_no_threshold = sorted(list(set(true_labels_str_no_threshold.unique()) | set(predicted_labels_str_no_threshold.unique())))
if 'unassigned' in cm_all_labels_no_threshold:
    cm_all_labels_no_threshold.remove('unassigned')
    cm_all_labels_no_threshold.append('unassigned')

print(f"Labels used for No Threshold Confusion Matrix: {cm_all_labels_no_threshold}")

cm_no_threshold = confusion_matrix(true_labels_str_no_threshold, predicted_labels_str_no_threshold, labels=cm_all_labels_no_threshold)
cm_df_no_threshold = pd.DataFrame(cm_no_threshold, index=cm_all_labels_no_threshold, columns=cm_all_labels_no_threshold)

plt.figure(figsize=(10, 8))
sns.heatmap(cm_df_no_threshold, annot=True, fmt='d', cmap='Blues', cbar_kws={'label': 'Number of Cells'})
plt.title('Confusion Matrix (pbmc3k - No Threshold)')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.tight_layout()
plt.show()

# Plot 2: Per-Class ROC AUC (Threshold = 0.0)
print("\n--- Per-Class ROC AUC (pbmc3k - No Threshold) ---")
if hasattr(scpred_model.classifier_, 'predict_proba') and y_pred_probs_df_no_threshold is not None:
    classes = scpred_model.classifier_.classes_
    y_true_binary = pd.get_dummies(true_labels_str_no_threshold, columns=classes, drop_first=False).values
    y_scores = y_pred_probs_df_no_threshold[[f"scpred_prob_{c}" for c in classes]].values

    plt.figure(figsize=(10, 8))
    for i, class_label in enumerate(classes):
        if y_true_binary.shape[1] > i and len(np.unique(y_true_binary[:, i])) > 1:
            fpr, tpr, _ = roc_curve(y_true_binary[:, i], y_scores[:, i])
            roc_auc = auc(fpr, tpr)
            plt.plot(fpr, tpr, label=f'Class {class_label} (AUC = {roc_auc:.2f})')
        else:
            print(f"Skipping ROC plot for Class {class_label}: Not enough unique true labels.")
    plt.plot([0, 1], [0, 1], 'k--', label='Chance (AUC = 0.50)')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('Receiver Operating Characteristic (ROC) Curve (pbmc3k - No Threshold)')
    plt.legend(loc="lower right")
    plt.grid(True)
    plt.show()

# Plot 3: Distribution of Prediction Probabilities (Threshold = 0.0)
print("\n--- Distribution of Prediction Probabilities (pbmc3k - No Threshold) ---")
if hasattr(scpred_model.classifier_, 'predict_proba') and y_pred_probs_df_no_threshold is not None:
    prob_df = query_adata_pred_no_threshold_filtered_for_umap.obs[[f"scpred_prob_{c}" for c in scpred_model.classifier_.classes_]].copy()
    prob_df['true_cell_type'] = query_adata_pred_no_threshold_filtered_for_umap.obs['cell_type'].astype(str)
    prob_melted = prob_df.melt(
        id_vars='true_cell_type',
        value_vars=[f"scpred_prob_{c}" for c in scpred_model.classifier_.classes_],
        var_name='predicted_class_prob_of',
        value_name='probability'
    )
    prob_melted['predicted_class_prob_of'] = prob_melted['predicted_class_prob_of'].str.replace('scpred_prob_', '')
    for true_type in sorted(prob_melted['true_cell_type'].unique()):
        plt.figure(figsize=(10, 5))
        subset_df = prob_melted[prob_melted['true_cell_type'] == true_type]
        sns.boxplot(data=subset_df, x='predicted_class_prob_of', y='probability', hue='predicted_class_prob_of', palette='viridis', legend=False)
        plt.title(f'Prediction Probabilities for True Cell Type: {true_type} (pbmc3k - No Threshold)')
        plt.xlabel('Probability of being Predicted as Class')
        plt.ylabel('Probability')
        plt.ylim(-0.05, 1.05)
        plt.grid(axis='y', linestyle='--', alpha=0.7)
        plt.xticks(rotation=45, ha='right')
        plt.show()

# UMAP plots (Threshold = 0.0)
if 'X_umap' in query_adata_pred_no_threshold_filtered_for_umap.obsm:
    print("\n--- UMAP: True vs Predicted Labels (pbmc3k - No Threshold) ---")
    axes_list = sc.pl.umap(
        query_adata_pred_no_threshold_filtered_for_umap,
        color=['cell_type', 'scpred_prediction'],
        title=['True Labels (Louvain Clusters)', 'scPred Predictions (No Threshold)'],
        frameon=False, show=False, ncols=2, wspace=0.3
    )
    fig = axes_list[0].figure
    fig.set_size_inches(12, 6)
    fig.suptitle('UMAP of Query Data: True vs Predicted Labels (pbmc3k - No Threshold)', y=1.02, fontsize=14)
    fig.tight_layout(rect=[0, 0.03, 1, 0.98])
    plt.show()

    print("\n--- UMAP: Misclassified Cells (pbmc3k - No Threshold) ---")
    query_adata_pred_no_threshold_filtered_for_umap.obs['misclassified'] = (
        true_labels_str_no_threshold != predicted_labels_str_no_threshold
    ).astype(str).astype('category')
    misclassified_palette = {'True': 'red', 'False': 'lightgray'}
    
    fig, ax = plt.subplots(1, 1, figsize=(8, 6))
    sc.pl.umap(
        query_adata_pred_no_threshold_filtered_for_umap,
        color='misclassified', palette=misclassified_palette, size=50, alpha=0.7,
        title='UMAP of Query Data: Misclassified Cells (pbmc3k - No Threshold)',
        frameon=False, show=False, legend_loc='upper right',
        ax=ax
    )
    fig.tight_layout()
    plt.show()

    print("\n--- UMAP: Prediction Confidence (pbmc3k - No Threshold) ---")
    if hasattr(scpred_model.classifier_, 'predict_proba'):
        def get_predicted_prob(row, classes):
            pred_class = row['scpred_prediction']
            if pred_class == 'unassigned' or pd.isna(pred_class):
                return np.nan
            prob_col_name = f'scpred_prob_{pred_class}'
            if prob_col_name in row.index:
                return row[prob_col_name]
            return np.nan
        prob_cols = [col for col in query_adata_pred_no_threshold_filtered_for_umap.obs.columns if col.startswith('scpred_prob_')]
        if all(col in query_adata_pred_no_threshold_filtered_for_umap.obs.columns for col in prob_cols):
            temp_obs_for_apply = query_adata_pred_no_threshold_filtered_for_umap.obs[['scpred_prediction'] + prob_cols]
            query_adata_pred_no_threshold_filtered_for_umap.obs['predicted_prob_score'] = temp_obs_for_apply.apply(
                lambda row: get_predicted_prob(row, scpred_model.classifier_.classes_), axis=1
            )
            fig, ax = plt.subplots(1, 1, figsize=(8, 6))
            sc.pl.umap(
                query_adata_pred_no_threshold_filtered_for_umap,
                color='predicted_prob_score', cmap='viridis',
                title='UMAP of Query Data: Prediction Confidence (pbmc3k - No Threshold)',
                frameon=False, vmin=0.0, vmax=1.0, show=False,
                ax=ax
            )
            fig.tight_layout()
            plt.show()
    else:
        print("Skipping UMAP by prediction confidence: Classifier does not support `predict_proba`.")
else:
    print("Skipping UMAP plots as 'X_umap' is not available in .obsm.")


# ==============================================================================
# THOROUGH ANALYSIS: SCENARIO 2 - WITH THRESHOLD (threshold=0.8)
# ==============================================================================
print("\n\n======== THOROUGH ANALYSIS: SCENARIO 2 - WITH THRESHOLD (threshold=0.8) ========")

# --- 4b. Predicting with the scPred Model (Threshold = 0.8) ---
print("\n--- Step 4b: Predicting on Query Data (Threshold = 0.8) ---")
query_adata_pred_with_threshold = scpred_model.predict(query_adata_raw.copy(), threshold=0.8) # Pass a copy

print("\nQuery Data with Predictions (With Threshold, first 5 rows):")
print(query_adata_pred_with_threshold.obs[['cell_type', 'scpred_prediction']].head())
print(f"\nQuery data .obs keys after prediction: {query_adata_pred_with_threshold.obs.keys().tolist()}")
print(f"Query data .obsm keys after prediction: {list(query_adata_pred_with_threshold.obsm.keys())}")
print("\nPredicted label distribution (With Threshold):\n", query_adata_pred_with_threshold.obs['scpred_prediction'].value_counts(dropna=False))

# Save the preprocessed query data with predictions
preprocessed_query_with_threshold_path = os.path.join(PREPROCESSED_DATA_DIR, 'pbmc3k_query_pred_with_threshold.h5ad')
query_adata_pred_with_threshold.write(preprocessed_query_with_threshold_path)
print(f"Preprocessed query data with predictions (With Threshold) saved to: {preprocessed_query_with_threshold_path}")


# --- 5b. Evaluating Predictions (Threshold = 0.8) ---
true_labels_with_threshold = query_adata_pred_with_threshold.obs['cell_type']
predicted_labels_with_threshold = query_adata_pred_with_threshold.obs['scpred_prediction']

# Filter out NaN predictions for evaluation to avoid ValueError
valid_cells_with_threshold = predicted_labels_with_threshold.notna()
true_labels_with_threshold_filtered = true_labels_with_threshold[valid_cells_with_threshold]
predicted_labels_with_threshold_filtered = predicted_labels_with_threshold[valid_cells_with_threshold]

pred_prob_cols_with_threshold = [col for col in query_adata_pred_with_threshold.obs.columns if col.startswith('scpred_prob_')]
y_pred_probs_df_with_threshold = None
if len(pred_prob_cols_with_threshold) > 0:
    y_pred_probs_df_with_threshold = query_adata_pred_with_threshold.obs.loc[valid_cells_with_threshold, pred_prob_cols_with_threshold].copy()
    y_pred_probs_df_with_threshold = y_pred_probs_df_with_threshold.fillna(0.0)


print("\n--- Evaluation for With Threshold Model ---")
metrics_results_with_threshold = _analysis_utils.evaluate_and_report_metrics(
    true_labels=true_labels_with_threshold_filtered,
    predicted_labels=predicted_labels_with_threshold_filtered,
    classifier_classes=scpred_model.classifier_.classes_,
    y_pred_probs=y_pred_probs_df_with_threshold
)


# --- 6b. Visualizing Results (Threshold = 0.8) ---
print("\n--- Step 6b: Visualizing Results (Threshold = 0.8) ---")

# For UMAP plots, we need the AnnData object to be filtered to only valid cells or handle NaNs in plots.
query_adata_pred_with_threshold_filtered_for_umap = query_adata_pred_with_threshold[valid_cells_with_threshold, :].copy()

if 'iroot' in query_adata_pred_with_threshold_filtered_for_umap.uns:
    del query_adata_pred_with_threshold_filtered_for_umap.uns['iroot']
if 'neighbors' in query_adata_pred_with_threshold_filtered_for_umap.uns:
    del query_adata_pred_with_threshold_filtered_for_umap.uns['neighbors']


if 'X_scpred_pca' in query_adata_pred_with_threshold_filtered_for_umap.obsm and 'X_umap' not in query_adata_pred_with_threshold_filtered_for_umap.obsm:
    print("Computing neighbors and UMAP based on X_scpred_pca for visualization...")
    sc.pp.neighbors(query_adata_pred_with_threshold_filtered_for_umap, n_neighbors=10, use_rep='X_scpred_pca', random_state=RANDOM_STATE)
    sc.tl.umap(query_adata_pred_with_threshold_filtered_for_umap, random_state=RANDOM_STATE)
    print(f"Query data .obsm keys after UMAP: {list(query_adata_pred_with_threshold_filtered_for_umap.obsm.keys())}")
elif 'X_umap' not in query_adata_pred_with_threshold_filtered_for_umap.obsm:
    print("X_scpred_pca or X_umap not found in .obsm. UMAP plots might not be generated.")

# Plot 1: Confusion Matrix (Threshold = 0.8)
print("\n--- Confusion Matrix (Threshold = 0.8) ---")
true_labels_str_with_threshold = true_labels_with_threshold_filtered.astype(str)
predicted_labels_str_with_threshold = predicted_labels_with_threshold_filtered.astype(str)

cm_all_labels_with_threshold = sorted(list(set(true_labels_str_with_threshold.unique()) | set(predicted_labels_str_with_threshold.unique())))
if 'unassigned' in cm_all_labels_with_threshold:
    cm_all_labels_with_threshold.remove('unassigned')
    cm_all_labels_with_threshold.append('unassigned')

print(f"Labels used for With Threshold Confusion Matrix: {cm_all_labels_with_threshold}")

cm_with_threshold = confusion_matrix(true_labels_str_with_threshold, predicted_labels_str_with_threshold, labels=cm_all_labels_with_threshold)
cm_df_with_threshold = pd.DataFrame(cm_with_threshold, index=cm_all_labels_with_threshold, columns=cm_all_labels_with_threshold)

plt.figure(figsize=(10, 8))
sns.heatmap(cm_df_with_threshold, annot=True, fmt='d', cmap='Blues', cbar_kws={'label': 'Number of Cells'})
plt.title('Confusion Matrix (pbmc3k - With Threshold=0.8)')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.tight_layout()
plt.show()

# Plot 2: Per-Class ROC AUC (Threshold = 0.8)
print("\n--- Per-Class ROC AUC (pbmc3k - With Threshold=0.8) ---")
if hasattr(scpred_model.classifier_, 'predict_proba') and y_pred_probs_df_with_threshold is not None:
    classes = scpred_model.classifier_.classes_
    y_true_binary = pd.get_dummies(true_labels_str_with_threshold, columns=classes, drop_first=False).values
    y_scores = y_pred_probs_df_with_threshold[[f"scpred_prob_{c}" for c in classes]].values

    plt.figure(figsize=(10, 8))
    for i, class_label in enumerate(classes):
        if y_true_binary.shape[1] > i and len(np.unique(y_true_binary[:, i])) > 1:
            fpr, tpr, _ = roc_curve(y_true_binary[:, i], y_scores[:, i])
            roc_auc = auc(fpr, tpr)
            plt.plot(fpr, tpr, label=f'Class {class_label} (AUC = {roc_auc:.2f})')
        else:
            print(f"Skipping ROC plot for Class {class_label}: Not enough unique true labels.")
    plt.plot([0, 1], [0, 1], 'k--', label='Chance (AUC = 0.50)')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('Receiver Operating Characteristic (ROC) Curve (pbmc3k - With Threshold=0.8)')
    plt.legend(loc="lower right")
    plt.grid(True)
    plt.show()

# Plot 3: Distribution of Prediction Probabilities (Threshold = 0.8)
print("\n--- Distribution of Prediction Probabilities (pbmc3k - With Threshold=0.8) ---")
if hasattr(scpred_model.classifier_, 'predict_proba') and y_pred_probs_df_with_threshold is not None:
    prob_df = query_adata_pred_with_threshold_filtered_for_umap.obs[[f"scpred_prob_{c}" for c in scpred_model.classifier_.classes_]].copy()
    prob_df['true_cell_type'] = query_adata_pred_with_threshold_filtered_for_umap.obs['cell_type'].astype(str)
    prob_melted = prob_df.melt(
        id_vars='true_cell_type',
        value_vars=[f"scpred_prob_{c}" for c in scpred_model.classifier_.classes_],
        var_name='predicted_class_prob_of',
        value_name='probability'
    )
    prob_melted['predicted_class_prob_of'] = prob_melted['predicted_class_prob_of'].str.replace('scpred_prob_', '')
    for true_type in sorted(prob_melted['true_cell_type'].unique()):
        plt.figure(figsize=(10, 5))
        subset_df = prob_melted[prob_melted['true_cell_type'] == true_type]
        sns.boxplot(data=subset_df, x='predicted_class_prob_of', y='probability', hue='predicted_class_prob_of', palette='viridis', legend=False)
        plt.title(f'Prediction Probabilities for True Cell Type: {true_type} (pbmc3k - With Threshold=0.8)')
        plt.xlabel('Probability of being Predicted as Class')
        plt.ylabel('Probability')
        plt.ylim(-0.05, 1.05)
        plt.grid(axis='y', linestyle='--', alpha=0.7)
        plt.xticks(rotation=45, ha='right')
        plt.show()

# UMAP plots (Threshold = 0.8)
if 'X_umap' in query_adata_pred_with_threshold_filtered_for_umap.obsm:
    print("\n--- UMAP: True vs Predicted Labels (pbmc3k - With Threshold=0.8) ---")
    axes_list = sc.pl.umap(
        query_adata_pred_with_threshold_filtered_for_umap,
        color=['cell_type', 'scpred_prediction'],
        title=['True Labels (Louvain Clusters)', 'scPred Predictions (With Threshold=0.8)'],
        frameon=False, show=False, ncols=2, wspace=0.3
    )
    fig = axes_list[0].figure
    fig.set_size_inches(12, 6)
    fig.suptitle('UMAP of Query Data: True vs Predicted Labels (pbmc3k - With Threshold=0.8)', y=1.02, fontsize=14)
    fig.tight_layout(rect=[0, 0.03, 1, 0.98])
    plt.show()

    print("\n--- UMAP: Misclassified Cells (pbmc3k - With Threshold=0.8) ---")
    query_adata_pred_with_threshold_filtered_for_umap.obs['misclassified'] = (
        true_labels_str_with_threshold != predicted_labels_str_with_threshold
    ).astype(str).astype('category')
    misclassified_palette = {'True': 'red', 'False': 'lightgray'}
    
    fig, ax = plt.subplots(1, 1, figsize=(8, 6))
    sc.pl.umap(
        query_adata_pred_with_threshold_filtered_for_umap,
        color='misclassified', palette=misclassified_palette, size=50, alpha=0.7,
        title='UMAP of Query Data: Misclassified Cells (pbmc3k - With Threshold=0.8)',
        frameon=False, show=False, legend_loc='upper right',
        ax=ax
    )
    fig.tight_layout()
    plt.show()

    print("\n--- UMAP: Prediction Confidence (pbmc3k - With Threshold=0.8) ---")
    if hasattr(scpred_model.classifier_, 'predict_proba'):
        def get_predicted_prob(row, classes):
            pred_class = row['scpred_prediction']
            if pred_class == 'unassigned' or pd.isna(pred_class):
                return np.nan
            prob_col_name = f'scpred_prob_{pred_class}'
            if prob_col_name in row.index:
                return row[prob_col_name]
            return np.nan
        prob_cols = [col for col in query_adata_pred_with_threshold_filtered_for_umap.obs.columns if col.startswith('scpred_prob_')]
        if all(col in query_adata_pred_with_threshold_filtered_for_umap.obs.columns for col in prob_cols):
            temp_obs_for_apply = query_adata_pred_with_threshold_filtered_for_umap.obs[['scpred_prediction'] + prob_cols]
            query_adata_pred_with_threshold_filtered_for_umap.obs['predicted_prob_score'] = temp_obs_for_apply.apply(
                lambda row: get_predicted_prob(row, scpred_model.classifier_.classes_), axis=1
            )
            fig, ax = plt.subplots(1, 1, figsize=(8, 6))
            sc.pl.umap(
                query_adata_pred_with_threshold_filtered_for_umap,
                color='predicted_prob_score', cmap='viridis',
                title='UMAP of Query Data: Prediction Confidence (pbmc3k - With Threshold=0.8)',
                frameon=False, vmin=0.0, vmax=1.0, show=False,
                ax=ax
            )
            fig.tight_layout()
            plt.show()
    else:
        print("Skipping UMAP by prediction confidence: Classifier does not support `predict_proba`.")
else:
    print("Skipping UMAP plots as 'X_umap' is not available in .obsm.")

print("\n--- Analysis Complete ---")
